# Week 2, Lab 2 — Tools with `@function_tool`


In [ ]:
WEEK = 'Week 2'
LAB = 'Lab 2 — function tools'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn openai openai-agents
else:
    %pip install -q ollama openai openai-agents


In [ ]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


In [ ]:
from agents import function_tool

@function_tool
def calculator_tool(expression: str) -> str:
    """Evaluate a basic arithmetic expression."""
    return calculator(expression)

@function_tool
def lookup_fact_tool(topic: str) -> str:
    """Look up a local fact about agentic AI."""
    return lookup_fact(topic)

agent = Agent(
    name="ToolTutor",
    instructions="Use tools for math and for course-topic facts. Be brief.",
    model=model,
    tools=[calculator_tool, lookup_fact_tool],
)

r1 = await Runner.run(agent, "What is 45 * 12 + 30?")
print("MATH:", r1.final_output)
r2 = await Runner.run(agent, "What is MCP?")
print("FACT:", r2.final_output)


## Exercise\n\nAdd `today_date` as a third tool. If the small model skips tools, insist in instructions: you MUST call a tool.\n\n**Next:** handoffs.
